# Sprint Changelog Generator — Before & After
### Companion notebook for *How Resourceful Is Your AI Skill?* (Part 3)

**→ [Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

Every team ships a sprint and then writes a summary manually in Confluence or Notion. This notebook builds a `sprint-changelog-generator` skill — then shows what happens when you don't enforce contracts on it.

The skill takes merged PR titles and descriptions → produces a human-readable 'what shipped this week' changelog. Simple premise. Three failure modes that are easy to miss.

| Gap | Fix | What you measure |
|---|---|---|
| No input filter | Skip `chore:` and `docs:` PRs | Token count: all PRs vs. relevant PRs |
| No output schema | Enforce `## Features`, `## Fixes`, `## Infrastructure` | Section presence: inconsistent vs. guaranteed |
| No audience contract | Parameterise `engineering` vs. `stakeholder` output | Token delta + tone consistency |

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

## Before you run anything — read this

### Step 1: Create a free Groq account
1. Go to [console.groq.com](https://console.groq.com) and sign up — **free, no credit card required**
2. Navigate to **API Keys** in the left sidebar
3. Click **Create API Key** → give it a name → copy the key

### Step 2: Add your key to Colab Secrets
1. Click the **🔑 key icon** in the left sidebar
2. Click **+ Add new secret**
3. Name: `GROQ_API_KEY` (exact spelling), Value: paste your key
4. Toggle **Notebook access** to ON

> ⚠️ Never paste your API key directly into a code cell. Always use Colab Secrets.

> ⚠️ LLMs are non-deterministic — re-run a cell if your output looks different from the post. The patterns hold even when exact text varies.

In [ ]:
%pip install openai --quiet

In [ ]:
# ── Provider config ───────────────────────────────────────────────────────────
# Default: Groq (free, no credit card). To switch providers, update BASE_URL + API_KEY.
#
# OpenAI:  BASE_URL = "https://api.openai.com/v1"   secret: OPENAI_API_KEY
# Kimi:    BASE_URL = "https://api.moonshot.cn/v1"  secret: KIMI_API_KEY
#
BASE_URL = "https://api.groq.com/openai/v1"
#
# ── Model options on Groq (free tier) ────────────────────────────────────────
# llama-3.1-8b-instant     → Fastest. Good for quick runs.
# llama-3.3-70b-versatile  → Stronger reasoning. Better schema compliance contrast.
#
MODEL = "llama-3.3-70b-versatile"
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import userdata
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=userdata.get("GROQ_API_KEY"))
print(f"Client ready. Model: {MODEL}")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "output":        response.choices[0].message.content,
        "input_tokens":  response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens":  response.usage.prompt_tokens + response.usage.completion_tokens,
    }

# ── Mock sprint dataset ───────────────────────────────────────────────────────
# 12 PRs from a realistic two-week sprint.
# type prefix follows Conventional Commits: feat / fix / chore / docs
SPRINT_PRS = [
    {"id": "PR-101", "type": "feat",     "title": "Add user timezone support",
     "description": "Users can now set their preferred timezone in account settings. All timestamps in emails and dashboards respect this preference."},
    {"id": "PR-102", "type": "fix",      "title": "Fix null pointer in billing module",
     "description": "Resolves a crash when a user's subscription record has no payment method attached. Adds a graceful fallback to the free tier."},
    {"id": "PR-103", "type": "feat",     "title": "Export usage reports to CSV",
     "description": "Admins can now download monthly usage reports as CSV from the admin panel. Includes per-user and per-team breakdowns."},
    {"id": "PR-104", "type": "chore",    "title": "Upgrade Node.js to v20 LTS",
     "description": "Bumps the base Docker image and CI pipeline to Node 20. No functional changes."},
    {"id": "PR-105", "type": "fix",      "title": "Correct pagination on search results",
     "description": "Search results were skipping every 11th result due to an off-by-one error in the offset calculation."},
    {"id": "PR-106", "type": "feat",     "title": "Slack notification integration",
     "description": "Teams can now connect a Slack webhook to receive alerts when their usage exceeds a configured threshold."},
    {"id": "PR-107", "type": "docs",     "title": "Update API authentication guide",
     "description": "Rewrites the OAuth2 setup section to reflect the new PKCE flow. Adds code samples in Python and TypeScript."},
    {"id": "PR-108", "type": "chore",    "title": "Extract shared config into separate package",
     "description": "Moves ESLint and Prettier configuration into a shared `@company/config` package for consistency across repos."},
    {"id": "PR-109", "type": "fix",      "title": "Handle empty state on dashboard widgets",
     "description": "Widgets no longer show a blank panel when there is no data; they now show a contextual empty-state message."},
    {"id": "PR-110", "type": "feat",     "title": "Two-factor authentication via TOTP",
     "description": "Users can enrol a TOTP authenticator app (Google Authenticator, Authy) for two-factor login. Recovery codes are generated on enrolment."},
    {"id": "PR-111", "type": "fix(security)", "title": "Patch IDOR vulnerability in document share links",
     "description": "Share link tokens were sequential integers and therefore guessable. Replaced with UUIDs. All existing share links have been invalidated."},
    {"id": "PR-112", "type": "docs",     "title": "Add runbook for on-call engineers",
     "description": "Documents the 5 most common alert types and their remediation steps."},
]

print(f"Dataset loaded: {len(SPRINT_PRS)} PRs")
print("Types:", {pr['type'] for pr in SPRINT_PRS})

---

## Gap 1: No Input Filter — Loading Everything Regardless of Relevance

The default invocation loads all merged PRs into the prompt and asks the model to summarise them.

`chore:` PRs (dependency bumps, config changes) and `docs:` PRs (README updates, runbooks) are not user-facing changes — they add tokens but contribute nothing to a changelog a user or stakeholder would read.

The fix: filter PRs by type before building the prompt. Load only `feat:` and `fix:` types. Token cost should drop in proportion to the noise removed.

**Experiment:** Build the prompt with all 12 PRs, then with only the relevant 8. Compare token counts.

In [ ]:
def format_prs_for_prompt(prs):
    lines = []
    for pr in prs:
        lines.append(f"- [{pr['id']}] {pr['title']}: {pr['description']}")
    return "\n".join(lines)

def build_changelog_prompt(prs, audience="engineering", enforce_schema=False):
    pr_block = format_prs_for_prompt(prs)
    schema_instruction = ""
    if enforce_schema:
        schema_instruction = """
Structure your response using EXACTLY these three section headings (use them even if a section is empty):
## Features
## Fixes
## Infrastructure

Do not use any other headings or categories."""

    audience_instruction = (
        "Write for a technical engineering audience: include PR IDs, technical detail, and impact."
        if audience == "engineering"
        else "Write for a non-technical stakeholder audience: plain English, business impact only, no PR IDs or code terms."
    )

    return f"""You are a technical writer. Generate a sprint changelog from these merged pull requests.

{audience_instruction}
{schema_instruction}

Merged PRs:
{pr_block}"""

# BEFORE: all 12 PRs, no filter
prompt_all = build_changelog_prompt(SPRINT_PRS, audience="engineering", enforce_schema=False)
before = ask(prompt_all)

print("=== BEFORE — all 12 PRs (no filter) ===")
print(before["output"])
print(f"\nTotal tokens: {before['total_tokens']} (input: {before['input_tokens']}, output: {before['output_tokens']})")

In [ ]:
# AFTER: filter to feat: and fix: only
RELEVANT_TYPES = {"feat", "fix", "fix(security)"}
relevant_prs = [pr for pr in SPRINT_PRS if pr["type"] in RELEVANT_TYPES]

prompt_filtered = build_changelog_prompt(relevant_prs, audience="engineering", enforce_schema=False)
after_filter = ask(prompt_filtered)

print(f"=== AFTER — {len(relevant_prs)} relevant PRs (chore: and docs: filtered) ===")
print(after_filter["output"])
print(f"\nTotal tokens: {after_filter['total_tokens']} (input: {after_filter['input_tokens']}, output: {after_filter['output_tokens']})")

saved   = before["total_tokens"] - after_filter["total_tokens"]
pct     = round((1 - after_filter["total_tokens"] / before["total_tokens"]) * 100)
print(f"\nDelta: {saved} tokens saved ({pct}% reduction) by removing {len(SPRINT_PRS) - len(relevant_prs)} non-user-facing PRs")

### What just happened

The token reduction comes entirely from not loading descriptions of PRs the output never references.

This is a routing problem applied at the input level. The skill had no concept of 'relevant input' — it accepted everything and trusted the model to ignore what didn't matter. That trust costs tokens every invocation.

At sprint velocity — 20 PRs, 30 PRs, a busy quarter — this compounds. A `chore:` filter is not clever engineering; it is a declared input contract that the Level 1 skill was missing.

---

## Gap 2: No Output Schema — The Model Invents Its Own Structure

Without a schema contract, the model is free to group changes however it sees fit.

Run the same prompt three times and you are likely to see three different heading structures: `### New Features`, `### Enhancements`, `### Improvements`, `### Updates`, `### Bug Fixes`, `### Patches`. Every variation breaks a downstream template that expects consistent headings.

The fix: enforce exactly three section headings — `## Features`, `## Fixes`, `## Infrastructure` — and verify programmatically that all three are present.

**Experiment:** Run without schema enforcement three times and check heading consistency. Then enforce the schema and verify.

In [ ]:
import re

def extract_headings(text):
    return re.findall(r'^#{1,3} .+', text, re.MULTILINE)

# BEFORE: run 3 times without schema enforcement, observe heading variance
print("=== BEFORE — 3 runs, no schema enforcement ===")
before_runs = []
for i in range(3):
    result = ask(build_changelog_prompt(relevant_prs, enforce_schema=False))
    headings = extract_headings(result["output"])
    before_runs.append(headings)
    print(f"\nRun {i+1} headings: {headings}")

# Check if all 3 runs produce identical structure
all_same = len(set(str(r) for r in before_runs)) == 1
print(f"\nConsistent structure across 3 runs: {all_same}")

In [ ]:
REQUIRED_SECTIONS = ["## Features", "## Fixes", "## Infrastructure"]

def check_schema(output):
    return {section: (section in output) for section in REQUIRED_SECTIONS}

# AFTER: run 3 times with schema enforcement
print("=== AFTER — 3 runs, schema enforced ===")
after_runs = []
all_pass = True
for i in range(3):
    result = ask(build_changelog_prompt(relevant_prs, enforce_schema=True))
    compliance = check_schema(result["output"])
    after_runs.append(compliance)
    status = "PASS" if all(compliance.values()) else "FAIL"
    if not all(compliance.values()):
        all_pass = False
    print(f"\nRun {i+1}: {compliance} → {status}")
    if i == 0:
        print(result["output"])  # print first run for visual inspection

print(f"\nAll 3 runs schema-compliant: {all_pass}")

### What just happened

Before the schema contract, each run is a coin flip on whether the heading names match what downstream tooling expects.

This is the schema compliance failure mode in its simplest form: the skill works — the content is correct — but the structure is non-deterministic. Any downstream system (a Confluence template, a Slack formatter, a weekly email pipeline) that parses `## Features` will break on `### New Features` or `### Enhancements`.

A schema contract in the prompt is not foolproof — the model can still deviate. The programmatic check (`check_schema`) is what makes the deviation visible. Without the check, you would never know a run failed until someone reported the broken template.

---

## Gap 3: No Audience Contract — One Output for Two Different Readers

An engineering changelog and a stakeholder summary are not the same document. Engineers want PR IDs, technical context, and impact on the system. Stakeholders want plain-English business value — no PR numbers, no code terms, no jargon.

A skill with no audience contract produces a single output and implicitly assumes one reader. In practice, it usually produces a technical document that a non-technical stakeholder cannot use — so someone still writes the stakeholder version manually.

The fix: an `audience` parameter that produces two distinct outputs from the same input, with a token delta that shows the difference in output density.

**Experiment:** Generate engineering and stakeholder changelogs from the same filtered PR set. Compare outputs and token usage.

In [ ]:
# BEFORE: single undeclared audience — assume engineering
result_before = ask(build_changelog_prompt(relevant_prs, enforce_schema=True))
print("=== BEFORE — single output, no audience contract ===")
print(result_before["output"])
print(f"\nTokens: {result_before['total_tokens']}")
print("Audience: undefined — stakeholder still has to write their own version manually")

In [ ]:
# AFTER: parameterised audience, two distinct outputs
result_eng  = ask(build_changelog_prompt(relevant_prs, audience="engineering",  enforce_schema=True))
result_stk  = ask(build_changelog_prompt(relevant_prs, audience="stakeholder",  enforce_schema=True))

print("=== AFTER — Engineering changelog ===")
print(result_eng["output"])
print(f"\nTokens: {result_eng['total_tokens']}")

print("\n" + "="*60)
print("=== AFTER — Stakeholder changelog ===")
print(result_stk["output"])
print(f"\nTokens: {result_stk['total_tokens']}")

print(f"\nDelta: stakeholder version uses {result_stk['output_tokens'] - result_eng['output_tokens']:+d} output tokens vs. engineering version")
print("Schema compliance:")
print(f"  Engineering: {check_schema(result_eng['output'])}")
print(f"  Stakeholder: {check_schema(result_stk['output'])}")

### What just happened

The same input, the same schema, two distinct outputs — distinguished by a single parameter.

The stakeholder version is typically shorter (fewer output tokens) because it strips technical detail. The engineering version is denser because it includes PR IDs and system-level context.

More importantly: the audience contract forces the skill author to make a decision that was previously implicit. A Level 1 skill with no `audience` parameter implicitly serves engineers. Every time a non-technical stakeholder receives that output, someone manually translates it. That cost is invisible in the skill spec but very visible in the team's weekly calendar.

---

## Delta Summary

In [ ]:
pct_filter = round((1 - after_filter["total_tokens"] / before["total_tokens"]) * 100)
schema_before_consistent = len(set(str(r) for r in before_runs)) == 1
schema_after_consistent  = all_pass

print(f"""
┌────────────────────────────┬──────────────────────────┬──────────────────────────────────┐
│ Metric                     │ Before (Level 1)         │ After (Level 2)                  │
├────────────────────────────┼──────────────────────────┼──────────────────────────────────┤
│ Token cost (full sprint)   │ {before['total_tokens']:<24} │ {after_filter['total_tokens']:<32} │
│ Token reduction            │ baseline                 │ ~{pct_filter}% (non-user-facing PRs filtered) │
│ Heading structure          │ {'Consistent' if schema_before_consistent else 'Inconsistent across runs':<24} │ {'Consistent: all 3 runs pass' if schema_after_consistent else 'Still failing — re-run':<32} │
│ Required sections present  │ Not enforced             │ Verified per run                 │
│ Audience contract          │ None — single output     │ engineering / stakeholder param  │
│ Downstream parseability    │ Unreliable               │ Guaranteed schema                │
└────────────────────────────┴──────────────────────────┴──────────────────────────────────┘
""")

### Maturity re-assessment

Without these three changes, `sprint-changelog-generator` is a Level 0 skill — a prompt snippet with no contract and no owner.

With them, it moves to **Level 2 — Verified Skill**:
- **Typed input boundary** — only `feat:`/`fix:` PRs accepted; others explicitly excluded
- **Output schema** — three required sections, verified programmatically per run
- **Audience contract** — declared parameter, two distinct output modes
- **Token budget** — input cost is predictable and bounded by sprint velocity, not total PR count

What's still missing for Level 3: continuous evaluation against real sprint data, a regression gate that fires when schema compliance drops below 100% after a model update, and a signed release with a changelog of contract changes.

---

## ✏️ Explore Further

Replace the PR list below with your own sprint's merged PRs. Run all three gaps and observe:
- How many of your PRs are `chore:` or `docs:` — and what that costs in tokens
- Whether the schema holds consistently on your sprint data
- Whether the stakeholder version would actually replace manual writing for your team

In [ ]:
# ── Try it yourself ──────────────────────────────────────────────────────────
# Replace the list below with your own sprint PRs.
# Format: {"id": "PR-NNN", "type": "feat|fix|chore|docs", "title": "...", "description": "..."}
# ─────────────────────────────────────────────────────────────────────────────
MY_SPRINT_PRS = [
    {"id": "PR-201", "type": "feat",  "title": "Add dark mode toggle",
     "description": "Users can now switch between light and dark themes in the settings panel. Preference is persisted per account."},
    {"id": "PR-202", "type": "fix",   "title": "Fix broken link in onboarding flow",
     "description": "The 'skip for now' link on step 3 of onboarding was redirecting to a 404. Fixed to redirect to the dashboard."},
    {"id": "PR-203", "type": "chore", "title": "Upgrade ESLint to v9",
     "description": "No functional changes. Resolves deprecation warnings in the CI pipeline."},
]

my_relevant = [pr for pr in MY_SPRINT_PRS if pr["type"] in RELEVANT_TYPES]
print(f"Relevant PRs: {len(my_relevant)} / {len(MY_SPRINT_PRS)}")

result = ask(build_changelog_prompt(my_relevant, audience="stakeholder", enforce_schema=True))
print("\nStakeholder changelog:")
print(result["output"])
print(f"\nSchema: {check_schema(result['output'])}")

---

## What's next

This notebook showed the schema compliance and token budget failure modes. The companion notebooks in this series cover different dominant dimensions:

- **[profile-bio Before & After](https://colab.research.google.com/github/SriharshaCR/blogs/blob/main/assets/notebooks/how-resourceful-is-your-ai-skill/03-hands-on/01_profile_bio_before_after.ipynb)** — token budget + data handling + behavioral regression
- **[Dependency Risk Assessor Before & After](https://colab.research.google.com/github/SriharshaCR/blogs/blob/main/assets/notebooks/how-resourceful-is-your-ai-skill/03-hands-on/03_dependency_risk_before_after.ipynb)** — routing accuracy + label enum enforcement

→ **[Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

*Part of the [open-skills](https://github.com/SriharshaCR/open-skills) project — AI skills built and shared openly.*